# 週末の買いタイミング判定：ANA HD（9202）・あおぞら銀行（8304）

毎週末（金曜の取引終了後〜日曜）に上から順に実行し、**翌週月曜の寄りで買うかどうか**を判断する材料をそろえるノートブックです。

**判定の型（これまでの検証と同じ）**
- 週の最終営業日の終値で **RSI(2) < 10** なら「シグナルあり」
- 翌週の最初の営業日の寄りで買い、その週の最終営業日の引けで売る

**同時に見る条件**（2008〜2026年の検証で、2銘柄と全銘柄で向きがそろったもの）

| 向き | 条件 |
|---|---|
| ＋（上昇確率が上がる側） | 25日線乖離率 ≤ -10%、週間騰落率 ≤ -10%、RSI(14) < 30、市場全体が週間 -3% 以下 |
| －（上昇確率が下がる側） | 75日線が上向き、終値 > 200日線、市場全体は上昇でこの銘柄だけ下落 |
| ？（1銘柄だけで目立った） | 業種平均より週間で5%以上下落、4日以上連続で下落 |

**参考点** ＝ ＋の条件の数 − －の条件の数（？は数えない）

> ⚠️ **注意**
> - どの条件も、多数の比較をしたことを補正すると統計的に確かとは言えません。参考点も、売買ルールとしての検証は済んでいません（下の「参考点別の過去の成績」は、条件を選んだのと同じデータでの集計です）。
> - データは Yahoo Finance（非公式）です。対象は今上場している銘柄だけなので、成績が良く見えやすい偏り（生存者バイアス）があります。
> - 決算発表などのイベントは考慮していません。発注前に各社のIRやTDnetで確認してください。
> - このノートブックは判断材料をそろえるもので、売買の推奨ではありません。

## 1. 設定

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# 週末に最新データへ更新するなら True（全約490銘柄を取り直すため数分かかる）
REFRESH = True
# 実際に寄りで買った後に損切り・利確を出し直すときは買値を入れる（例: {"9202.T": 2850.0}）
ENTRY_PRICE = {}
RECENT_YEARS = 3  # 過去シグナル一覧に表示する年数

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from weekpick import data as WD
from weekpick import timing as T
from weekpick.universe import load_universe

# 日本語フォント（見つかったものを使う）
for f in ["IPAGothic", "IPAexGothic", "Hiragino Sans", "Yu Gothic", "Meiryo", "Noto Sans CJK JP"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
TICKERS = list(T.TARGETS)

## 2. データ更新

市場全体の値動きや業種平均を使うため、2銘柄だけでなく TOPIX500 相当の全銘柄を読み込みます。
銘柄一覧（`data/universe_jpx.csv`）が古くなったら `python -m weekpick universe` で更新してください。

In [ ]:
u = load_universe("topix500")
if REFRESH:
    failed = WD.fetch(list(u["ticker"]), refresh=True)
    print("取得失敗:", failed or "なし")
p = WD.load_panel(list(u["ticker"]))
for n in p.notes:
    print("・", n)
last = p.close.index[-1]
print(f"最新データ: {last:%Y-%m-%d}（{'月火水木金土日'[last.weekday()]}）")
if last.weekday() != 4:
    print("⚠️ 最新データが金曜ではありません。週の途中のデータで判定しています（祝日で木曜が週最終日の場合は問題ありません）。")

## 3. 今週末の判定

In [ ]:
ind = T.indicators(p, u.set_index("ticker")["sector"])
now = T.latest(ind, TICKERS)
now.insert(0, "銘柄", [T.TARGETS[t] for t in now.index])
num = ["RSI2", "RSI14", "25日線乖離%", "週間騰落%", "市場の週間騰落%", "業種平均との差%"]
now[num] = now[num].round(1)
now.T

In [ ]:
for t, r in now.iterrows():
    if not r["シグナル"]:
        print(f"■ {r['銘柄']}: シグナルなし（RSI2={r['RSI2']}）→ 見送り")
        continue
    on = [c for c in T.CONDITIONS if r[c.label]]
    print(f"■ {r['銘柄']}: シグナルあり（RSI2={r['RSI2']}）  参考点 {int(r['参考点']):+d}")
    for c in on:
        print(f"   [{c.side}] {c.label} … {c.note}")
    if not on:
        print("   同時に出ている条件はありません")

## 4. 過去のシグナル時の成績

### 4-1. 条件ごとの上昇確率（2008年〜、銘柄別）
「あり」はその条件が同時に出ていたシグナル週、「なし」は出ていなかったシグナル週です。回数が少ない（目安として20回未満）ものは、確率が大きくぶれます。

In [ ]:
hist = T.history(p, ind, TICKERS)
base = hist.groupby("ticker")["翌週騰落率"].agg(回数="size", 上昇確率=lambda r: (r > 0).mean(), 平均=lambda r: r.mean())
base.index = [T.TARGETS[t] for t in base.index]
display(base.style.format({"上昇確率": "{:.1%}", "平均": "{:+.2%}"}))

ct = T.conditional_table(hist)
ct["銘柄"] = ct["ticker"].map(T.TARGETS)
ct = ct.drop(columns="ticker").set_index(["銘柄", "条件"])
ct.style.format({"あり_上昇確率": "{:.1%}", "なし_上昇確率": "{:.1%}"}, na_rep="—")

### 4-2. 参考点別の過去の成績
⚠️ 条件を選んだのと同じ期間での集計です。将来もこの通りになる保証はありません。

In [ ]:
sc = hist.groupby(["ticker", "参考点"])["翌週騰落率"].agg(回数="size", 上昇確率=lambda r: (r > 0).mean(), 平均=lambda r: r.mean())
sc.index = sc.index.set_levels([[T.TARGETS[t] for t in sc.index.levels[0]], sc.index.levels[1]])
sc.style.format({"上昇確率": "{:.1%}", "平均": "{:+.2%}"})

### 4-3. 直近のシグナル週の一覧

In [ ]:
since = last - pd.DateOffset(years=RECENT_YEARS)
recent = hist[hist["判定日"] >= since].copy()
recent["銘柄"] = recent["ticker"].map(T.TARGETS)
label = {c.key: f"[{c.side}]{c.label}" for c in T.CONDITIONS}
cols = ["銘柄", "判定日", "買い", "売り", "RSI2", "翌週騰落率", "参考点"] + list(label)
view = recent[cols].rename(columns=label)
for c in ["判定日", "買い", "売り"]:
    view[c] = view[c].dt.date
for t, g in recent.groupby("ticker"):
    r = g["翌週騰落率"]
    print(f"{T.TARGETS[t]}: 直近{RECENT_YEARS}年 {len(r)}回, 上昇確率 {(r > 0).mean():.1%}, 平均 {r.mean():+.2%}")
view.style.format({"RSI2": "{:.1f}", "翌週騰落率": "{:+.2%}"}).map(
    lambda v: "background-color: #e6f4ea" if v is True else "", subset=list(label.values()))

## 5. チャート（直近1年）
▲ はシグナル週の判定日です。

In [ ]:
fig, axes = plt.subplots(len(TICKERS) * 2, 1, figsize=(11, 4.2 * len(TICKERS)), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1] * len(TICKERS)})
start = last - pd.DateOffset(years=1)
for i, t in enumerate(TICKERS):
    c = p.close[t].loc[start:]
    ax, ax2 = axes[2 * i], axes[2 * i + 1]
    ax.plot(c.index, c, color="black", lw=1.2, label="終値")
    for n, col in [(25, "tab:blue"), (75, "tab:orange"), (200, "tab:green")]:
        ax.plot(c.index, p.close[t].rolling(n).mean().loc[start:], lw=0.9, color=col, label=f"{n}日線")
    ev = hist[(hist["ticker"] == t) & (hist["判定日"] >= start)]
    ax.scatter(ev["判定日"], c.reindex(ev["判定日"]), marker="^", s=60,
               c=np.where(ev["翌週騰落率"] > 0, "tab:red", "tab:gray"), zorder=3, label="シグナル（赤=翌週上昇）")
    ax.set_title(f"{T.TARGETS[t]}（{t}）")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    ax2.plot(c.index, ind["rsi2"][t].loc[start:], lw=0.8, color="tab:purple")
    ax2.axhline(T.RSI2_THRESHOLD, color="red", lw=0.8, ls="--")
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI(2)")
    ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 損切り・利確の目安（損失：利益 = 1：2）

- **A**：直近10日の安値（買値の方が低ければ買値）から ATR×0.1 下で損切り
- **B**：買値から ATR×2 下で損切り

利確は、どちらも損失幅の2倍上です。買値は、寄りで約定するまでは最新の終値を仮に使います。約定後は、1. の `ENTRY_PRICE` に買値を入れて再実行してください。

⚠️ 損切り・利確を入れた場合の成績はまだ検証していません。1週間で売る運用では、多くの取引が損切りにも利確にも届かず、時間切れで決済されます。窓を開けて損切りラインを飛び越えた場合は、ラインより不利な価格で約定します。

In [ ]:
lv = pd.DataFrame({T.TARGETS[t]: T.trade_levels(p, t, ENTRY_PRICE.get(t)) for t in TICKERS}).T
lv.style.format("{:.1f}")

## 7. 判断のメモ（例）

- シグナルがなければ見送る
- シグナルがあれば、参考点と、出ている条件の過去の「あり」回数・上昇確率を確認する（回数が少ない条件は重視しない）
- 「－」の条件（上昇トレンド中の軽い押し目、この銘柄だけの下げ）が重なるときは慎重に
- 発注前に、翌週の決算発表・権利落ち・開示を確認する

結果は毎週記録しておくと、条件の効き方を今後のデータで確認できます（今のところ、本当の意味での確認は今後のデータでしかできません）。